In [1]:
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from cuml.cluster import HDBSCAN


/home/gpuuser7/gpuuser7_a/prateek/LLM_with_ads/.lmarena-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'cuda.core.experimental'

In [2]:
# ============================================================
# 1. Configuration
# ============================================================

INPUT_FILE = "/home/gpuuser7/gpuuser7_a/prateek/LLM_with_ads/data/processed/lmarena/dataset/arena_preference_en_single_turn.parquet"
OUTPUT_FILE = "lmarena_clustered.csv"
EMBEDDINGS_FILE = "/home/gpuuser7/gpuuser7_a/prateek/LLM_with_ads/data/processed/embeddings/lmarena_query_embeddings_embeddinggemma_300m.npz"

QUERY_COLUMN = "query"

MODEL_NAME = "google/embeddinggemma-300m"
# EmbeddingGemma task prompt for clustering (applied in encode via prompt_name)
PROMPT_NAME = "Clustering"

BATCH_SIZE = 64
MIN_CLUSTER_SIZE = 30


In [3]:
# ============================================================
# 2. Check GPU
# ============================================================

if torch.cuda.is_available():
    device = "cuda"
    print("Using GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    device = "cpu"
    print("CUDA GPU not available. Using CPU.")

Using GPU: NVIDIA A100-SXM4-80GB
CUDA version: 12.6


In [4]:
# ============================================================
# 3. Load dataset
# ============================================================
df = pd.read_parquet(INPUT_FILE)

print("Number of rows:", len(df))

# Remove missing queries
df = df.dropna(subset=[QUERY_COLUMN]).copy()

# Raw query strings — EmbeddingGemma prompts are applied in encode()
queries = df[QUERY_COLUMN].astype(str).tolist()

print("Number of valid queries:", len(queries))


Number of rows: 36579
Number of valid queries: 36579


In [5]:
# ============================================================
# 4. Load embedding model (only if embeddings are missing)
# ============================================================

from pathlib import Path

_emb_path = Path(EMBEDDINGS_FILE)
if _emb_path.exists():
    model = None
    print(f"Cached embeddings found at {_emb_path} — skipping model load.")
else:
    model = SentenceTransformer(MODEL_NAME, device=device)
    print("Model loaded on:", model.device)


Cached embeddings found at /home/gpuuser7/gpuuser7_a/prateek/LLM_with_ads/data/processed/embeddings/lmarena_query_embeddings_embeddinggemma_300m.npz — skipping model load.


In [6]:
# ============================================================
# 5. Load embeddings if available, else generate
# ============================================================

from pathlib import Path

_emb_path = Path(EMBEDDINGS_FILE)
if _emb_path.exists():
    _data = np.load(_emb_path, allow_pickle=True)
    embeddings = _data["embeddings"]
    print("Loaded embeddings from:", _emb_path)
    print("Embedding shape:", embeddings.shape)
else:
    if model is None:
        model = SentenceTransformer(MODEL_NAME, device=device)
        print("Model loaded on:", model.device)

    print("Available prompts:", list(getattr(model, "prompts", {}) or {}))
    encode_kwargs = dict(
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    # Prefer named Clustering prompt; fall back to STS / encode_query if needed
    prompts = getattr(model, "prompts", None) or {}
    if PROMPT_NAME in prompts:
        encode_kwargs["prompt_name"] = PROMPT_NAME
    elif "STS" in prompts:
        encode_kwargs["prompt_name"] = "STS"
        print(f"{PROMPT_NAME!r} not found; using prompt_name='STS'")
    else:
        print("No named prompts found; encoding without prompt_name")

    embeddings = model.encode(queries, **encode_kwargs)
    print("Generated embedding shape:", embeddings.shape)

    _emb_path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        _emb_path,
        embeddings=embeddings,
        ids=df["id"].to_numpy() if "id" in df.columns else np.arange(len(embeddings)),
        model_name=np.array(MODEL_NAME),
        prompt_name=np.array(encode_kwargs.get("prompt_name", "")),
    )
    print("Saved embeddings to:", _emb_path)


Loaded embeddings from: /home/gpuuser7/gpuuser7_a/prateek/LLM_with_ads/data/processed/embeddings/lmarena_query_embeddings_embeddinggemma_300m.npz
Embedding shape: (36579, 768)


In [ ]:
# ============================================================
# 5b. (optional) Force re-save current embeddings
# ============================================================

from pathlib import Path

Path(EMBEDDINGS_FILE).parent.mkdir(parents=True, exist_ok=True)
np.savez_compressed(
    EMBEDDINGS_FILE,
    embeddings=embeddings,
    ids=df["id"].to_numpy() if "id" in df.columns else np.arange(len(embeddings)),
)
print("Saved embeddings to:", EMBEDDINGS_FILE)
print("Shape:", embeddings.shape)


In [ ]:
# ============================================================
# 6. HDBSCAN hyperparameter tuning (GPU via cuML)
# ============================================================

from itertools import product
from sklearn.metrics import silhouette_score

X = embeddings.astype(np.float32)

param_grid = {
    "min_cluster_size": [15, 30, 50, 80, 120],
    "min_samples": [5, 10, 15, 30],
    "cluster_selection_method": ["eom", "leaf"],
}

rows = []
print(f"Tuning HDBSCAN over {np.prod([len(v) for v in param_grid.values()])} configs...")

for min_cluster_size, min_samples, method in product(
    param_grid["min_cluster_size"],
    param_grid["min_samples"],
    param_grid["cluster_selection_method"],
):
    clusterer = HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        metric="euclidean",
        cluster_selection_method=method,
        prediction_data=False,
    )
    labels = np.asarray(clusterer.fit_predict(X))

    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = int((labels == -1).sum())
    noise_ratio = n_noise / len(labels)

    # Silhouette on non-noise points (needs >=2 clusters and enough samples)
    sil = np.nan
    mask = labels != -1
    if n_clusters >= 2 and mask.sum() >= n_clusters + 1:
        try:
            # subsample for speed if huge
            idx = np.flatnonzero(mask)
            if len(idx) > 8000:
                rng = np.random.default_rng(42)
                idx = rng.choice(idx, size=8000, replace=False)
            sil = float(silhouette_score(X[idx], labels[idx], metric="euclidean"))
        except Exception:
            sil = np.nan

    # Prefer: more than a handful of clusters, not all noise, higher silhouette
    # Penalize extreme noise and trivial 0/1-cluster solutions
    if n_clusters < 2 or noise_ratio > 0.95:
        score = -1.0
    else:
        sil_term = 0.0 if np.isnan(sil) else sil
        # soft preference for mid-range cluster counts
        cluster_term = -abs(np.log10(max(n_clusters, 1)) - np.log10(20))
        score = sil_term - noise_ratio + 0.15 * cluster_term

    rows.append({
        "min_cluster_size": min_cluster_size,
        "min_samples": min_samples,
        "method": method,
        "n_clusters": n_clusters,
        "n_noise": n_noise,
        "noise_ratio": round(noise_ratio, 4),
        "silhouette": None if np.isnan(sil) else round(sil, 4),
        "score": round(score, 4),
    })
    sil_str = f"{sil:.4f}" if not np.isnan(sil) else "  nan"
    print(
        f"mcs={min_cluster_size:3d} ms={min_samples:2d} {method:4s} | "
        f"clusters={n_clusters:3d} noise={noise_ratio:6.1%} sil={sil_str} score={score:.4f}"
    )

tune_df = pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True)
print("\nTop configs:")
print(tune_df.head(10).to_string(index=False))

best = tune_df.iloc[0]
print("\nBest config:", best.to_dict())

best_clusterer = HDBSCAN(
    min_cluster_size=int(best["min_cluster_size"]),
    min_samples=int(best["min_samples"]),
    metric="euclidean",
    cluster_selection_method=best["method"],
    prediction_data=True,
)
cluster_labels = np.asarray(best_clusterer.fit_predict(X))
df["cluster"] = cluster_labels
print("Clusters found:", len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0))
print("Noise points:", int((cluster_labels == -1).sum()))


In [ ]:
# ============================================================
# 7. Cluster statistics
# ============================================================

cluster_counts = (
    df["cluster"]
    .value_counts()
    .sort_index()
)

print("\nCluster distribution:")
print(cluster_counts)

In [ ]:
# ============================================================
# 8. Save results
# ============================================================

df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\nSaved to:", OUTPUT_FILE)

In [ ]:
# ============================================================
# 9. Inspect all clusters
# ============================================================

N_SHOW = 8

for cid, n in df["cluster"].value_counts().sort_index().items():
    print("=" * 80)
    print(f"Cluster {cid} (n={n})")
    print("=" * 80)
    for i, q in enumerate(df.loc[df["cluster"] == cid, "query"].head(N_SHOW), 1):
        q = " ".join(str(q).split())
        # if len(q) > 180:
        #     q = q[:180] + "..."
        print(f"  {i}. {q}")
    print()


In [2]:
categories = pd.read_csv("/home/gpuuser7/gpuuser7_a/prateek/LLM_with_ads/data/processed/lmarena/lmarena_query_qwen3_categories.csv")

In [3]:
categories[['id','query','domain','intent', 'commercial_intent']].head()

,id,query,domain,intent,commercial_intent
0,c4b9710c-8d64-4bee-a0b0-94637ae4cc65,Compare Tormenta20 with DnD5e,Gaming,Product Comparison,2.0
1,4a4380bb-bbdb-495f-8e09-39a08d88a28f,What about 1 year in Sweden with proof e.g. tr...,Travel,Travel Planning,2.0
2,40f981ca-cca6-4e2a-a48c-d10d51884efe,What are Tricky the Clown's forms?,Entertainment,Information Seeking,0.0
3,e67468c9-69b7-4b84-9ec2-27879128c4c6,If I exercise until I feel that the muscle ach...,Sports & Fitness,Information Seeking,0.0
4,171f1847-cbf1-4c85-b16a-2ecb52b06d19,okay i am using you in intereview i will just ...,General,Guidance,0.0


In [4]:
print(len(categories['domain'].unique()))
print(len(categories['intent'].unique()))

2265
1843


In [13]:
labeled = categories.dropna(subset=["domain", "intent"])
domain_intent_counts = (
    labeled.groupby(["domain", "intent"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)
print("unique domain+intent pairs:", len(domain_intent_counts))
domain_intent_counts

unique domain+intent pairs: 6807


,domain,intent,count
0,Technology,Information Seeking,2831
1,Creative Writing,Creative,2808
2,Science,Information Seeking,1603
3,Technology,Product Research,1177
4,General Knowledge,Information Seeking,827
...,...,...,...
6802,Food,Personal Preference,1
6803,Food,Entertainment,1
6804,Food,Comparison,1
6805,Folklore & Mythology,Information Seeking,1


In [14]:
domains_sorted = sorted(set(categories['domain'].unique()))
pd.DataFrame({"domain": domains_sorted}).to_csv("domains_sorted.csv", index=False)
